This script loads and visualises a time series of Sentinel-2 imagery over the Boccassuolo landslide area using Google Earth Engine (GEE)
and geemap. It imports the Area of Interest (AOI) and the mapped landslide boundary, then displays RGB composites for several dates between
March and July 2025. The most recent image is shown by default, while earlier dates are available as optional layers for visual comparison.
The AOI and landslide polygons are displayed as transparent outlines to highlight the spatial extent of the analysis.
The map allows rapid inspection of temporal surface changes, useful for monitoring landslide evolution, identifying displacement patterns,
and visually supporting the kinematic interpretation presented in the related research.


In [1]:
import ee
import geemap

# Initialize Google Earth Engine
try:
    ee.Authenticate()
    ee.Initialize(project='ee-my-gmg-mapping') # Replace with your actual Earth Engine project ID
except Exception as e:
    print(f"An error occurred during Earth Engine initialization: {e}")
    print("Please ensure you have authenticated and provided a valid Earth Engine project ID.")

# ---------------------------------------------------------
# 2️⃣ Create Map
# ---------------------------------------------------------
Map = geemap.Map()

# ---------------------------------------------------------
# 3️⃣ Load AOI and Landslide Polygons
# ---------------------------------------------------------
aoi = ee.FeatureCollection('projects/ee-my-gmg-mapping/assets/Boccassuolo_AOI')
landslide = ee.FeatureCollection('projects/ee-my-gmg-mapping/assets/boccasuolo_boundary')

# ---------------------------------------------------------
# 4️⃣ Define List of Dates
# ---------------------------------------------------------
dates = [
    '2025-03-23',
    '2025-03-31',
    '2025-04-04',
    '2025-05-02',
    '2025-05-15',
    '2025-05-25',
    '2025-06-04',
    '2025-06-14',
    '2025-07-09'  # latest
]

# ---------------------------------------------------------
# 5️⃣ Visualization Parameters
# ---------------------------------------------------------
rgbVis = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.3
}

# ---------------------------------------------------------
# 6️⃣ Style AOI and Landslide Polygons
# ---------------------------------------------------------
styledAOI = aoi.style(
    color='red',
    width=2,
    fillColor='00000000'  # transparent
)

styledLandslide = landslide.style(
    color='blue',
    width=1,
    fillColor='00000000'
)

# ---------------------------------------------------------
# 7️⃣ Load and Add Sentinel-2 RGB Images
# ---------------------------------------------------------
for index, date in enumerate(dates):
    start = ee.Date(date)
    end = start.advance(1, 'day')

    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi)
          .filterDate(start, end)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)))

    img = s2.first()

    # Check if image exists
    if s2.size().getInfo() > 0:
        # Only the LAST date (latest image) is shown by default
        is_latest = (index == len(dates) - 1)

        Map.addLayer(
            img.clip(aoi),
            rgbVis,
            f"RGB {date}",
            shown=is_latest
        )
    else:
        print(f"No image available on {date}")

# ---------------------------------------------------------
# 8️⃣ Add AOI and Landslide Layers Last (On Top)
# ---------------------------------------------------------
Map.addLayer(styledAOI, {}, "AOI (outline)", True)
Map.addLayer(styledLandslide, {}, "Landslide Boundary", False)

# ---------------------------------------------------------
# 9️⃣ Center the Map on the AOI
# ---------------------------------------------------------
Map.centerObject(aoi, 14)

# Display map
Map

Map(center=[44.274037273066355, 10.607631102700688], controls=(WidgetControl(options=['position', 'transparent…